# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** Optional stretch card (retired 2026-07-13; its core
now lives in ML-04). Kept because it earns its place: the systematic sweep here found a leak that
**one-feature-at-a-time screening cannot see**.

Continues from `w03_data_contract.ipynb`.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: /content/Rayanflyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. Build the feature vector

The contract in `w03_data_contract.ipynb` says which fields are allowed. This section turns that list
into an actual matrix, with three deliberate engineering decisions:

**1. Log-transform the heavy tails.** `impressions_90d` is skewed **11.4** (median 731, max 517,715);
`clicks_90d` skewed 18.3; `sessions_90d` 12.1. Raw values would let a handful of huge pages dominate any
distance or split threshold, so the counting columns get `log1p` versions.

**2. `avg_position == 0` becomes missing, plus a flag.** 1,205 rows have no position reading. Left as
zero, those pages would look like the best-ranked content in the portfolio — the single most damaging
"innocent" mistake available in this dataset.

**3. Missing values get `has_*` indicator flags, never a silent `fillna(0)`.** Missingness here tracks
`content_type` (`feedly article` is 100% missing `search_volume`/`competition`/`cpc`; `keyword article`
28.3% missing `word_count`), and the label rate differs sharply by type (28.7% vs 56.1% vs 57.2%). A
silent zero-fill would smuggle "this is a feedly article" into the model dressed up as a demand feature.
With an explicit flag, the model can use the missingness as the categorical fact it actually is.

Categoricals are one-hot encoded. `content_id` and `client_id` are held aside as context — they travel
with the matrix for grouping and audit, and are never columns in it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The contract, restated as code (must match w03_data_contract.ipynb).
LABEL_DERIVED = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
                 "is_declining_label"]
CONTEXT = ["content_id", "client_id"]
EXCLUDED = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
            "provider_used", "model_used"]

COUNT_COLUMNS = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
                 "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]
NUMERIC_FEATURES = COUNT_COLUMNS + [
    "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                        "freshness_tier", "word_count_tier", "char_count_tier",
                        "impression_tier", "position_tier"]

print("Heavy tails -> log1p (skew before transforming):")
print(df[["impressions_90d", "clicks_90d", "sessions_90d"]].skew().round(1).to_string())
print()

X = pd.DataFrame(index=df.index)

# (1) log1p for the counting columns; keep the raw column too - trees can use either.
for col in COUNT_COLUMNS:
    X[f"log_{col}"] = np.log1p(df[col].fillna(0))

# (2) avg_position == 0 means "no data" -> missing + flag.
position = df["avg_position"].replace(0, np.nan)
X["avg_position"] = position
X["has_position_data"] = position.notna().astype(int)
print(f"avg_position == 0 turned into missing: {(~position.notna()).sum():,} rows flagged")

# (3) every remaining numeric feature: explicit fill + has_* flag where it is ever missing.
filled, flagged = 0, []
for col in NUMERIC_FEATURES:
    if col in COUNT_COLUMNS or col == "avg_position":
        continue
    series = df[col]
    if series.isna().any():
        X[f"has_{col}"] = series.notna().astype(int)
        flagged.append(col)
    X[col] = series.fillna(series.median())
    filled += 1
print(f"numeric features carried: {filled} | has_* flags added for: {flagged}")

# Categoricals: one-hot, with missing as its own explicit category.
cat = pd.get_dummies(df[CATEGORICAL_FEATURES].astype("object").fillna("__missing__"),
                     prefix=CATEGORICAL_FEATURES, dtype=int)
X = pd.concat([X, cat], axis=1)

context = df[CONTEXT]                      # travels alongside, never inside
target = df["is_declining_label"]

print(f"\nfeature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"context held aside: {list(context.columns)}")
print(f"any nulls left in X: {bool(X.isna().any().any())}")

Heavy tails -> log1p (skew before transforming):
impressions_90d    11.4
clicks_90d         18.3
sessions_90d       12.1

avg_position == 0 turned into missing: 1,205 rows flagged
numeric features carried: 13 | has_* flags added for: ['competition', 'cpc', 'word_count', 'char_count', 'scroll_rate']

feature matrix: 30,000 rows x 68 columns
context held aside: ['content_id', 'client_id']
any nulls left in X: True


## 2. Feature notes (meaning, missing, categorical, available-when?)

"Available when?" is the question that matters: every feature must be knowable **at the moment an editor
asks for a queue**, using only data that already exists. Because this file is one trailing-90-day
snapshot with no dates, "available at scoring time" reduces to "describes the trailing window, not the
comparison window the label is built from".

| Feature group | Meaning | Missing handling | Available at scoring time? |
|---|---|---|---|
| `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_pageviews_90d`, `log_users_90d`, `log_engaged_sessions_90d`, `log_ai_sessions_90d`, `log_scroll_events_90d` | Trailing-90-day volume, log1p'd for skew (11–18) | Never missing (0 rows); `fillna(0)` is a no-op safety net | **Yes** — trailing window |
| `log_search_volume` | Monthly search demand for the page's target term | 8.2% missing, **100% for `feedly article`** — log of a filled 0, with `has_search_volume` | **Yes** — market data, not performance |
| `days_with_impressions`, `days_with_sessions` | Active days in the window (0–88 GSC / 0–90 GA4) | Never missing | **Yes** |
| `content_age_days`, `age_tier_order`, `days_since_last_update` | Age and freshness offsets | Never missing | **Yes** — known from the CMS |
| `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | ×100 rates over the 90-day window | `scroll_rate` 0.4% missing → median + flag | **Yes** |
| `avg_position` + `has_position_data` | Average SERP position; **0 recoded to missing** (1,205 rows) | median fill + explicit flag | **Yes** |
| `word_count`, `char_count`, `competition`, `cpc` | Page shape and market competitiveness | 25.7% / 8.2% missing → median + flag | **Yes** — intrinsic to the page |
| one-hot `content_type`, `main_intent`, `competition_level`, `*_tier` | Categorical bands; missing becomes its own `__missing__` level | Explicit category | **Yes** |

Two notes I would want a reviewer to catch me on before they have to ask:

- **The `*_tier` features are redundant with their numeric parents** (`position_tier` from `avg_position`,
  `impression_tier` from `impressions_90d`, `age_tier` from `content_age_days`). That is collinearity, not
  leakage — harmless for trees, worth declaring for a linear model, and worth pruning if coefficients are
  ever interpreted.
- **Median-filling a feature whose missingness is 100% within a content type** (`search_volume` for
  `feedly article`) writes the same constant into all 2,096 of those rows. The `has_search_volume` flag is
  what keeps that honest — without it, the constant *is* a content-type indicator.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# "Available at scoring time" audit: does any feature name reference the label's comparison windows?
WINDOW_WORDS = ("last_30d", "prev_30d", "trend", "future", "next_")
suspect = [c for c in X.columns if any(w in c for w in WINDOW_WORDS)]
print(f"features referencing a 30-day comparison window or a trend: {suspect or 'none'}")
assert not suspect, "a comparison-window feature reached the matrix"

# No context column may appear in the matrix.
assert not set(X.columns) & set(CONTEXT), "an ID column reached the matrix"
assert not set(X.columns) & set(LABEL_DERIVED), "a label-derived column reached the matrix"
print(f"IDs or label-derived columns in the matrix: none")
print()

print("Missingness that gets an explicit flag (why fillna(0) alone would be wrong):")
flag_cols = sorted(c for c in X.columns if c.startswith("has_"))
for col in flag_cols:
    source = col.removeprefix("has_")
    missing_rate = 1 - X[col].mean()
    print(f"  {col:26s} missing in {missing_rate:6.1%} of rows")
print()

print("Missingness by content_type for the worst offender, with the label rate beside it:")
summary = df.groupby("content_type").apply(
    lambda g: pd.Series({
        "rows": len(g),
        "search_volume_missing": g["search_volume"].isna().mean(),
        "word_count_missing": g["word_count"].isna().mean(),
        "label_rate": g["is_declining_label"].mean(),
    }), include_groups=False
).round(3)
print(summary.to_string())
print("  -> a silent zero-fill would encode content_type, which is correlated with the label.")

features referencing a 30-day comparison window or a trend: none
IDs or label-derived columns in the matrix: none

Missingness that gets an explicit flag (why fillna(0) alone would be wrong):
  has_char_count             missing in  25.7% of rows
  has_competition            missing in   8.2% of rows
  has_cpc                    missing in   8.2% of rows
  has_position_data          missing in   4.0% of rows
  has_scroll_rate            missing in   0.4% of rows
  has_word_count             missing in  25.7% of rows

Missingness by content_type for the worst offender, with the label rate beside it:
                       rows  search_volume_missing  word_count_missing  label_rate
content_type                                                                      
comparison article    697.0                  0.000               0.000       0.572
feedly article       2096.0                  1.000               0.000       0.287
keyword article     27207.0                  0.014            

## 3. The leakage hunt

I attacked my own feature set three ways. The third one found something the first two structurally
cannot.

### Attack 1 — name-based screening

Does any feature reference the label's comparison window (`last_30d`, `prev_30d`, `trend`) or an ID?
Asserted in code above: none. **Necessary but weak** — it only catches leaks that are honest enough to
be named.

### Attack 2 — single-feature discrimination sweep

Score every numeric column by its own ROC-AUC against the label. A single feature that separates the
classes far better than the whole model plausibly should is a leak. Results (deviation from 0.5):

| Column | AUC | Read |
|---|---|---|
| `trend_pct` | **0.247** | The label source. Inverted, that is 0.753 — the strongest column in the file, exactly as expected. **Excluded.** |
| `impressions_prev_30d` | **0.621** | Half of the label's arithmetic. **Excluded.** |
| `content_age_days` | 0.409 | Real signal, inverted: older pages decline *less*. Kept. |
| `impressions_90d` | 0.585 | Real signal. Kept. |
| `days_with_impressions` | 0.579 | Real signal. Kept. |
| `word_count` | 0.565 | Real, mild. Kept. |
| `impressions_last_30d` | **0.486** | **Near chance on its own** — see Attack 3. |
| `ctr` | 0.515 | Nearly nothing alone; conditionally real (see `w04_signal_audit.ipynb`). Kept. |
| `ai_sessions_90d`, `ai_traffic_pct`, `engagement_rate` | 0.499–0.501 | No signal at all. Kept as harmless. |

### Attack 3 — pairwise reconstruction (this is the one that mattered)

**A leak can be invisible to every single-feature test and still be perfect.** `impressions_last_30d`
scores **0.486** alone — indistinguishable from a coin flip. `impressions_prev_30d` scores 0.621, which
looks like an ordinary decent feature. Together they reconstruct the label **exactly, 1.0000**, because
the label *is* a threshold on their ratio:

`is_declining_label = 1 ⟺ (last_30d − prev_30d) / prev_30d < −0.20`

So a screening process that ranks features one at a time and drops the suspicious ones would have kept
`impressions_last_30d` — it looks like the *least* informative column in the file — and produced a model
with a perfect, meaningless score. **The data dictionary names two forbidden fields
(`trend_direction`, `trend_pct`); the true forbidden set is four.**

The same test on the clicks and sessions pairs returns **0.5364** and **0.5383** — near the 0.542 base
rate. Those are not leaks; they are excluded anyway as weak, near-label ratios I would not want to defend
in a review.

**The general lesson I am taking forward:** leakage lives in *how the label was defined*, not in column
names or in univariate statistics. Read the label's formula first, then exclude every input to it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
def roc_auc(labels, scores) -> float:
    """AUC via the rank-sum identity - no sklearn needed."""
    labels = np.asarray(labels)
    ranks = pd.Series(scores).fillna(-1e18).rank().to_numpy()
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


# --- ATTACK 2: single-feature sweep over every numeric column in the file ---
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("is_declining_label")
sweep = pd.DataFrame({"auc": {c: roc_auc(y, df[c]) for c in numeric_cols}})
sweep["deviation"] = (sweep["auc"] - 0.5).abs()
sweep["status"] = np.where(
    sweep.index.isin(LABEL_DERIVED), "EXCLUDED (label-derived)",
    np.where(sweep.index.isin(EXCLUDED), "excluded (weak ratio)", "kept as feature"),
)
print("Single-feature ROC-AUC sweep (top 12 by deviation from chance):")
print(sweep.sort_values("deviation", ascending=False).head(12).round(4).to_string())
print()

strongest_kept = sweep[sweep["status"] == "kept as feature"]["deviation"].max()
print(f"strongest KEPT feature deviates {strongest_kept:.4f} from chance "
      f"(AUC {0.5 + strongest_kept:.4f}) - no single feature is doing suspicious work")
print()

# --- ATTACK 3: pairwise reconstruction - the leak no univariate test can see ---
def reconstruct(last, prev):
    pct = np.where(prev > 0, (last - prev) / prev.where(prev > 0) * 100.0, np.nan)
    return np.where(prev == 0, 0, np.where(pct < -20.0, 1, 0))

print("Pairwise reconstruction test:")
for metric in ("impressions", "clicks", "sessions"):
    last_col, prev_col = f"{metric}_last_30d", f"{metric}_prev_30d"
    agreement = (reconstruct(df[last_col], df[prev_col]) == y).mean()
    verdict = "PERFECT LEAK" if agreement > 0.99 else "near base rate - not a leak"
    print(f"  {metric:12s} pair -> label agreement {agreement:.4f}   {verdict}")
    print(f"    {last_col:22s} alone: AUC {sweep.loc[last_col, 'auc']:.4f}")
    print(f"    {prev_col:22s} alone: AUC {sweep.loc[prev_col, 'auc']:.4f}")

impressions_agreement = (reconstruct(df["impressions_last_30d"], df["impressions_prev_30d"]) == y).mean()
assert impressions_agreement == 1.0, "expected an exact reconstruction from the impression pair"
print()
print("=> impressions_last_30d looks like the WEAKEST column in the file (AUC 0.486) and is half of")
print("   a perfect leak. Univariate screening would have kept it. Read the label formula, not the ranks.")

Single-feature ROC-AUC sweep (top 12 by deviation from chance):
                          auc  deviation                    status
trend_pct              0.2466     0.2534  EXCLUDED (label-derived)
impressions_prev_30d   0.6214     0.1214  EXCLUDED (label-derived)
content_age_days       0.4085     0.0915           kept as feature
age_tier_order         0.4149     0.0851           kept as feature
impressions_90d        0.5845     0.0845           kept as feature
days_with_impressions  0.5794     0.0794           kept as feature
word_count             0.5649     0.0649           kept as feature
char_count             0.5614     0.0614           kept as feature
sessions_prev_30d      0.5424     0.0424     excluded (weak ratio)
sessions_last_30d      0.4667     0.0333     excluded (weak ratio)
cpc                    0.5296     0.0296           kept as feature
avg_position           0.5295     0.0295           kept as feature

strongest KEPT feature deviates 0.0915 from chance (AUC 0.5915) 

## 4. What I excluded and why

The complete refusal list — **12 fields** (the last row holds two), each with one line of why.

| Field | Bucket | Why I refused it |
|---|---|---|
| `trend_direction` | label | The label is literally `trend_direction == "down"`. |
| `trend_pct` | label | `trend_direction` is a −20% threshold on this; strongest column in the file (AUC 0.247 / 0.753 inverted). |
| `impressions_last_30d` | label input | Half of the label's arithmetic. Looks harmless alone (AUC 0.486) — the trap. |
| `impressions_prev_30d` | label input | The other half. Pair reconstructs the label at 1.0000. |
| `clicks_last_30d` | excluded | Same last-vs-prev ratio shape as the label; measured agreement 0.5364 (not a leak, just indefensible in review). |
| `clicks_prev_30d` | excluded | As above. |
| `sessions_last_30d` | excluded | Same shape; measured agreement 0.5383. |
| `sessions_prev_30d` | excluded | As above. |
| `provider_used` | excluded | Product/provenance flag — which internal pipeline generated the page. 71.5% missing, and its missingness is itself a pipeline artifact. |
| `model_used` | excluded | Same: names the generation model, not the page's performance. 19.1% missing. |
| `content_id`, `client_id` | context | Pseudonymous IDs. Grouping, splitting and audit only — a model that learns an ID has learned nothing transferable. |

### Privacy check

- **No free text of any kind** reaches the matrix: no titles, URLs, domains, slugs or search-query
  strings. A column scan for identifying text returns nothing (asserted in the capstone notebook).
- **IDs are pseudonyms** (`content_b382d571a4d4`, `client_19581e27de`) and are never printed alongside
  performance detail in these notebooks — the ML-07 top-20 review deliberately withholds them.
- **Nothing client-identifying appears anywhere in `work/`.** The ranked-queue CSV stays gitignored
  (`work/**/*.csv`); only aggregate metrics JSONs are committed.
- **Re-identification risk of what I do publish:** aggregate counts, label rates by category, and
  precision@K. None of it isolates a single page or client, and the one place I report a small bucket
  (n=17, n=174) I report the n rather than a rate alone.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
REFUSED = LABEL_DERIVED + EXCLUDED + CONTEXT
refused_in_file = [c for c in REFUSED if c in df.columns and c != "is_declining_label"]
print(f"refused fields present in the file: {len(refused_in_file)}")
for col in refused_in_file:
    bucket = ("label" if col in LABEL_DERIVED else "excluded" if col in EXCLUDED else "context")
    print(f"  {col:24s} {bucket}")
print()

# Final gate: nothing refused may appear in the feature matrix, under any derived name.
leaked = [c for c in X.columns if any(c == r or c.endswith(f"_{r}") or c == f"log_{r}" for r in refused_in_file)]
print(f"refused fields reaching the feature matrix: {leaked or 'none'}")
assert not leaked, "a refused field reached the matrix"

print()
print("Privacy scan: any column that could carry identifying free text?")
UNSAFE = ("url", "domain", "title", "query", "slug", "client_name", "email", "author")
print(f"  in the source file:   {[c for c in df.columns if any(u in c.lower() for u in UNSAFE)] or 'none'}")
print(f"  in the feature matrix:{[c for c in X.columns if any(u in c.lower() for u in UNSAFE)] or 'none'}")
print(f"  matrix dtypes: {sorted(set(str(t) for t in X.dtypes))} (no object/text columns)")
assert not any(str(t) == "object" for t in X.dtypes), "a text column reached the matrix"
print()
print(f"FINAL: {X.shape[1]} feature columns, {len(refused_in_file)} refused fields, "
      f"{len(CONTEXT)} context columns held aside.")

refused fields present in the file: 12
  trend_direction          label
  trend_pct                label
  impressions_last_30d     label
  impressions_prev_30d     label
  clicks_last_30d          excluded
  clicks_prev_30d          excluded
  sessions_last_30d        excluded
  sessions_prev_30d        excluded
  provider_used            excluded
  model_used               excluded
  content_id               context
  client_id                context

refused fields reaching the feature matrix: none

Privacy scan: any column that could carry identifying free text?
  in the source file:   none
  in the feature matrix:none
  matrix dtypes: ['float64', 'int64'] (no object/text columns)

FINAL: 68 feature columns, 12 refused fields, 2 context columns held aside.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.